# Занятие 4, пара 2. Форматы и открытые данные

Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai

**Как проходить тетрадь.** Основные примеры работают независимо от заданий «Ваш ход». Если пока пропускаете задание, пропустите и его ячейку `check(...)`; продолжайте со следующего обычного раздела. Упражнения могут использовать ответы предыдущих упражнений. Сохраните свои попытки в отдельной копии тетради.

## Файл скачали. Данные получили? Пока не факт

Сегодня одна и та же маленькая таблица будет представлена двумя способами: CSV и XML. Разберём, как текст превращается в столбцы и числа, почему `001` нельзя бездумно превращать в `1`, и что надо проверить после сохранения.

Потребуются знакомые таблицы Pandas, списки и словари. Новое — кодировка, разделитель, десятичный знак и дерево XML; всё разберём по одному действию. Три станции и температуры **вымышлены для занятия**, это не метеорологические наблюдения. Основная работа не требует сети и дополнительных библиотек для Excel/Parquet.

К концу пары у вас будут две одинаковые по смыслу таблицы, проверенная сохранённая копия и небольшой план получения данных для ЛР2.


### Подключим учебные файлы

Следующая ячейка — служебная настройка комплекта. Она находит папку `course_support` у текущей папки или выше и подключает данные и проверки. Её не нужно писать по памяти или менять для задания. `DATA_DIR` — путь к учебным данным, `WORKSPACE_DIR` — корень вашей рабочей папки. `check(номер, ответы)` сравнивает ответ с условиями упражнения, но не выставляет оценку.

Работайте в своей копии и запускайте ячейки сверху вниз. Переменные из другой тетради не нужны. Если поддержку не удалось найти, откройте тетрадь внутри полного комплекта, а не переносите один файл отдельно. Git и интернет для основной работы не нужны.


In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check

PROJECT_DIR


### 1. Почему файл может не читаться

`Path` уже подключён служебной ячейкой. Путь соединяем через `/`. У `read_text` параметр `encoding` говорит, как байты превратить в буквы. Наш CSV специально сохранён в Windows-1251, поэтому попытка прочитать его как UTF-8 даст ошибку. `try/except UnicodeDecodeError` здесь **намеренно** ловит именно её, чтобы занятие продолжилось; другие ошибки прятать не будем.


In [ ]:
import pandas as pd

csv_path = DATA_DIR / 'formats' / 'stations.csv'
try:
    text = csv_path.read_text(encoding='utf-8')
except UnicodeDecodeError:
    print('UTF-8 не подходит. Кодировка учебного файла — cp1251.')


Теперь зададим известную кодировку. Не надо угадывать её бесконечным перебором: сначала смотрим описание источника. `print` покажет текст до превращения в таблицу. Найдите символ между столбцами и символ внутри дробных чисел — они здесь разные.


In [ ]:
text = csv_path.read_text(encoding='cp1251')
print(text)


CSV — текстовая таблица. Обычно первая строка содержит названия столбцов, дальше — записи. Название формата исторически связано с запятыми, но в нашем файле столбцы разделены `;`, а запятая используется в числе `18,5`. «Расширение .csv» не даёт полной инструкции чтения. В описании набора отдельно указаны кодировка, разделитель и десятичный знак.


### 2. Исправляем по одному параметру

`pd.read_csv` читает файл в DataFrame. Сначала зададим только кодировку. По умолчанию разделитель — запятая, поэтому ожидаем неправильное разбиение. Это не авария Python: формально таблица получена, но смысл потерян. Посмотрим имена столбцов и первые строки.


In [ ]:
wrong = pd.read_csv(csv_path, encoding='cp1251')
print(wrong.columns)
wrong


`sep=';'` говорит, где заканчивается одно поле и начинается другое. Теперь столбцов три. Но температура ещё записана текстом, а идентификатор станций уже мог потерять ведущие нули. `dtypes` показывает типы столбцов — это свойство, скобки не нужны.


In [ ]:
with_separator = pd.read_csv(csv_path, encoding='cp1251', sep=';')
print(with_separator.dtypes)
with_separator


Добавим `decimal=','`: запятая внутри температуры означает дробную часть числа. Это параметр чтения чисел, а не разделитель столбцов. Проверьте тип температуры и попробуйте сложить значения: теперь мы хотим арифметику, а не склеивание текста.

Метод столбца `.mean()` считает среднее арифметическое; он нужен именно числовому столбцу.


In [ ]:
with_numbers = pd.read_csv(csv_path, encoding='cp1251', sep=';', decimal=',')
print(with_numbers.dtypes)
print(with_numbers['temperature_c'].mean())


Остаётся ID. Станция `001` — это название/код, а не количество станций. Для арифметики он не нужен. `dtype={'station_id': str}` задаёт тип **только указанному столбцу**: `str` означает строку. Остальные типы Pandas определит сам. Нули сохраняются при чтении исходного файла; перевод уже прочитанного числа `1` в строку не восстановит, было там `01` или `001`.


In [ ]:
stations = pd.read_csv(
    csv_path,
    encoding='cp1251',
    sep=';',
    decimal=',',
    dtype={'station_id': str}
)
print(stations.dtypes)
stations


### ✏️ Ваш ход — прочитать таблицу осознанно

Самостоятельно прочитайте исходный файл в `my_csv`, сохранив три столбца, ID с ведущими нулями и числовую температуру. В своём выводе объясните роль каждого из четырёх параметров. Затем посчитайте среднюю температуру методом `.mean()` — только как упражнение с вымышленными числами, без климатического вывода.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('04.2.1', my_csv)


**Мой вывод:** замените эту строку своим объяснением результата. Назовите конкретное число, поле или участок изображения, на который опираетесь.


### 3. Те же данные, другой способ записи

CSV удобен для плоской таблицы. XML хранит дерево элементов: внутри одного элемента могут находиться другие. У элемента есть имя — **тег**, текст и **атрибуты**, то есть значения внутри открывающего тега.

```xml
<station id="001">
    <city>Сыктывкар</city>
    <temperature>18.5</temperature>
</station>
```

Здесь `station` — элемент станции, `id` — его атрибут, а `city` и `temperature` — дочерние элементы. Это напоминает дерево HTML из прошлой пары, но сейчас мы разбираем XML средствами стандартной библиотеки. Синтаксис и правила разбора у XML свои.


`xml.etree.ElementTree` — встроенный модуль для XML; короткое имя `ET` просто удобное сокращение. `parse` читает файл и строит дерево. `getroot` возвращает корневой элемент. Сначала посмотрим только его: свойство `tag` хранит имя, `attrib` — словарь атрибутов.


In [ ]:
import xml.etree.ElementTree as ET

xml_path = DATA_DIR / 'formats' / 'stations.xml'
tree = ET.parse(xml_path)
root = tree.getroot()
print(root.tag)
print(root.attrib)


У корня метод `find('station')` находит **первый** дочерний элемент с таким тегом. У элемента `.get('id')` читает атрибут. Это не словарь JSON, хотя название метода знакомое. Если элемент не найден, `find` вернёт `None`; мы пока работаем с известным учебным файлом.


In [ ]:
station = root.find('station')
station_id = station.get('id')
print(station_id)


Получим город в два шага: сначала дочерний элемент, затем его `.text`. Температура тоже сначала текст. `float` превращает строку с десятичной точкой в число. В XML нашего источника точка; запятая была особенностью CSV, а не свойством температуры.


In [ ]:
city_element = station.find('city')
city = city_element.text
temperature_element = station.find('temperature')
temperature_text = temperature_element.text
temperature = float(temperature_text)
print(city)
print(temperature, type(temperature))


`findall('station')` вернёт все дочерние станции. С ними можно работать циклом `for`, как со списком. Сначала проверим количество и ID; таблицу соберём отдельным шагом.


In [ ]:
elements = root.findall('station')
print(len(elements))
for element in elements:
    print(element.get('id'))


Чтобы собрать таблицу, повторим схему из JSON: пустой список → одна запись → словарь строки → `append` → `DataFrame`. В каждой строке нужны ключи `station_id`, `city`, `temperature_c`. Извлечение одного значения мы только что разобрали.

Для произвольного источника сначала проверяют схему, наличие обязательных элементов и единицы. Если `find` вернул `None`, нужно разобраться с исходником, а не автоматически записывать температуру 0. Здесь пример мал и полностью описан; это не универсальный загрузчик недоверенного XML из интернета.


### ✏️ Ваш ход — собрать XML в таблицу

Создайте `my_xml`: три строки и столбцы в порядке `station_id`, `city`, `temperature_c`. Температуру преобразуйте в число. Сверьте значения с `my_csv`. Объясните, что в XML было атрибутом, а что текстом элемента.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('04.2.2', my_xml)


**Мой вывод:** замените эту строку своим объяснением результата. Назовите конкретное число, поле или участок изображения, на который опираетесь.


### 4. Сохраним — и проверим, что прочитается обратно

Создадим папку результатов. `mkdir` создаёт папку, `parents=True` — вместе с промежуточными, `exist_ok=True` разрешает повторный запуск. Исходники остаются отдельно. `to_csv` записывает таблицу; `index=False` не добавляет служебный индекс отдельным столбцом. Для своей новой копии выберем UTF-8, запятую между полями и точку внутри чисел — это стандартные параметры записи, но их надо указать в описании.


Для показа сохранения заново прочитаем исходный CSV с разобранными параметрами. `demo_csv` не зависит от ответов `my_csv` и `my_xml`.

In [ ]:
demo_csv = pd.read_csv(DATA_DIR / 'formats' / 'stations.csv',
                       sep=';', decimal=',', encoding='cp1251',
                       dtype={'station_id': 'string'})

In [ ]:
OUTPUT_DIR = WORKSPACE_DIR / 'reports' / 'formats'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_path = OUTPUT_DIR / 'stations_clean.csv'
demo_csv.to_csv(output_path, index=False, encoding='utf-8')
print(output_path.read_text(encoding='utf-8'))


CSV хранит текстовые значения, но не строгую схему типов Pandas. Поэтому при повторном чтении ID снова надо объявить строкой. Успешное сохранение ещё не доказывает, что другой человек прочитает файл правильно. Проверим повторную загрузку в новую переменную, не используя уже готовую таблицу вместо чтения.


### ✏️ Ваш ход — проверить сохранённую копию

Прочитайте `output_path` в `restored`, сохранив ID как строки. Проверка должна подтвердить исходные три записи. В выводе напишите параметры чтения вашей новой копии и сравните их с параметрами исходного файла.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('04.2.3', restored)


**Мой вывод:** замените эту строку своим объяснением результата. Назовите конкретное число, поле или участок изображения, на который опираетесь.


### 5. А Excel и Parquet? Короткое знакомство

Excel удобен человеку: листы, форматирование, формулы. Но красиво оформленный отчёт может содержать несколько таблиц, объединённые ячейки и примечания внутри данных. Перед анализом надо найти саму таблицу и единицы, а не просто вызвать функцию.

Parquet — двоичный столбцовый формат. Он хранит информацию о типах и может сжимать данные, поэтому полезен для аналитических таблиц. Для чтения/записи Pandas нужен дополнительный движок, например PyArrow. **На этой паре его не устанавливаем.** У маленького файла служебные данные могут занимать заметную часть размера; «Parquet всегда меньше CSV» — плохое правило. Формат выбирают под задачу и доступные инструменты.

Для проекта достаточно корректного CSV, JSON или другого доступного формата. Набор библиотек сам по себе не улучшает работу.


### 6. Перенос в ваш проект: начинаем ЛР2

Откройте файл `docs/ЛР2.md` в своей рабочей папке (папка `docs` находится рядом с `workbooks`), либо тот же файл в репозитории курса. Составьте короткий паспорт, можно прямо в следующей Markdown-ячейке:

- На какой вопрос я хочу ответить? Что будет одной строкой данных?
- Где источник и кто его опубликовал? Какие условия использования указаны?
- Как получить данные повторно: файл, API или разрешённый сбор страниц?
- Какие поля, единицы, кодировка и пропуски ожидаются? Какие проверки это подтвердят?
- Что я пока не знаю об источнике и что проверю следующим шагом?

Если тема ещё не выбрана, начните с учебного набора: задача сейчас — воспроизводимое получение и описание, а не масштаб. Открытый доступ и открытая лицензия — разные вещи. Если право на распространение исходника неясно, не публикуйте чужой набор целиком: сохраните ссылку и инструкцию получения, а условия обсудите с преподавателем.

Сохраните тетрадь и копию результата. GitHub необязателен: файлы/архив принимаются по тем же критериям. На следующем этапе будем искать дефекты уже внутри таблицы — пропуски, дубликаты, типы и подозрительные значения.

Источники для возвращения к теме: [Pandas — ввод и вывод](https://pandas.pydata.org/docs/user_guide/io.html), [ElementTree](https://docs.python.org/3/library/xml.etree.elementtree.html).


**Черновик паспорта моего источника**

Вопрос: …

Одна строка: …

Источник и условия: …

Получение и проверки: …

Ограничения: …
